In [ ]:
import sys

libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet"
}
for lib in libs.values():
    sys.path.append(lib)

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from hypernet import const

In [ ]:
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
path_to_model = "./saving/me_int/SavedModel/"
path_to_data = "./../datagen/data/test/"
path_to_figs = "./postproc/testing/"

In [ ]:
# Model
model = tf.keras.models.load_model(path_to_model)
# Model - Forward reaction rate
model_fcorr = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.layers[5].output
)

In [ ]:
def get_data(filename):
    # Collect data
    data = pd.read_csv(filename).to_numpy()
    *x, y = np.split(data, [3,6,13], axis=1)
    t = x[0][:,:1].squeeze()
    # Make prediction
    y_pred = model(x).numpy().squeeze()
    fcorr = model_fcorr(x)[0].numpy().squeeze()
    de = - fcorr * const.UKB * T / const.eV_to_J
    return t, y, y_pred, de

In [ ]:
def plot_data(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    n, d = y.shape if (x is not None) else yref.shape
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=n//20,
        ls='--',
        lw=1
    )
    for i in range(d):
        if (x is not None):
            ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
        if (xref is not None):
            ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
        ax.plot([], [], c=COLORS[i], label=names[i])
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.close()

In [ ]:
for (i, case) in enumerate(os.listdir(path_to_data)):
    filename = path_to_data + "/" + case
    t, y, y_pred, de = get_data(filename)
    path_to_figsi = path_to_figs + f"/case_" + str(i+1).zfill(3)
    os.makedirs(path_to_figsi, exist_ok=True)
    kwargs=dict(
        x=t[1:],
        y=de[1:],
        labels=[r'$t$ [s]', r'$\delta e$ [eV]'],
        names=[r'O$_2^{(%d)}$' % (i+1) for i in range(3)]
    )
    plot_data(
        scales=['log', 'linear'],
        figname=path_to_figsi + f"/fcorr.png",
        **kwargs
    )